# 🔍 FakeScope — Fake Engagement Detection
**Behavioural Analytics Hackathon | Problem Statement 3**

**Dataset:** airt-ml/twitter-human-bots (HuggingFace) | License: CC-BY-SA 3.0  
**Model:** Random Forest Classifier (scikit-learn)  

---
Run each cell **top to bottom**. At the end, download your model + plots.


## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install pyarrow pandas scikit-learn matplotlib joblib -q
print('✅ All packages ready')

## 📂 Step 2 — Download Real Dataset from HuggingFace

In [ ]:
import pandas as pd
import os

os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('models', exist_ok=True)

print('⬇️  Downloading dataset from HuggingFace...')

URL = 'https://huggingface.co/datasets/airt-ml/twitter-human-bots/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet'
df_raw = pd.read_parquet(URL)
df_raw.to_parquet('data/twitter_human_bots.parquet', index=False)

print(f'✅ Downloaded! Shape: {df_raw.shape}')
print(f'\n   Columns: {list(df_raw.columns)}')
print(f'\n   Label distribution:')
print(df_raw['account_type'].value_counts())
df_raw.head(3)

## 🔧 Step 3 — Feature Engineering
We engineer **23 behavioural features** from the raw account-level data.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = df_raw.copy()

# ── Target ───────────────────────────────────────────────
df['label'] = (df['account_type'] == 'bot').astype(int)

# ── Activity features ─────────────────────────────────────
df['tweets_per_day']     = df['average_tweets_per_day'].fillna(0).clip(0, 1000)
df['account_age_days']   = df['account_age_days'].fillna(0).clip(0, 6000)
df['total_tweets']       = df['statuses_count'].fillna(0).clip(0, 3_000_000)
df['total_likes_given']  = df['favourites_count'].fillna(0).clip(0, 1_000_000)
df['tweet_density']      = (df['total_tweets'] / (df['account_age_days'] + 1)).clip(0, 500)
df['likes_per_day']      = (df['total_likes_given'] / (df['account_age_days'] + 1)).clip(0, 1000)

# ── Network features ──────────────────────────────────────
df['followers_count']       = df['followers_count'].fillna(0).clip(0, 50_000_000)
df['friends_count']         = df['friends_count'].fillna(0).clip(0, 5_000_000)
df['follower_friend_ratio'] = (df['followers_count'] / (df['friends_count'] + 1)).clip(0, 500)
df['friends_to_followers']  = (df['friends_count'] / (df['followers_count'] + 1)).clip(0, 500)
df['likes_to_tweets']       = (df['total_likes_given'] / (df['total_tweets'] + 1)).clip(0, 500)

# ── Profile completeness ──────────────────────────────────
df['has_description']     = df['description'].notna().astype(int)
df['description_length']  = df['description'].fillna('').str.len().clip(0, 280)
df['has_location']        = df['location'].notna().astype(int)
df['has_default_pic']     = df['default_profile_image'].astype(int)
df['is_default_profile']  = df['default_profile'].astype(int)
df['is_verified']         = df['verified'].astype(int)
df['has_geo']             = df['geo_enabled'].astype(int)
df['profile_completeness']= (df['has_description'] + df['has_location'] +
                              (1 - df['has_default_pic']) + (1 - df['is_default_profile']) + df['has_geo'])

# ── Anomaly flags ─────────────────────────────────────────
df['is_high_tweeter']      = (df['tweets_per_day'] > 50).astype(int)
df['abnormal_tweet_rate']  = (df['tweets_per_day'] > 100).astype(int)
df['sudden_fame']          = ((df['account_age_days'] < 30) & (df['followers_count'] > 1000)).astype(int)
df['engagement_paradox']   = ((df['followers_count'] > 500) & (df['total_likes_given'] == 0)).astype(int)

# ── Final feature set ─────────────────────────────────────
FEATURES = [
    'tweets_per_day', 'account_age_days', 'total_tweets', 'total_likes_given',
    'tweet_density', 'likes_per_day', 'followers_count', 'friends_count',
    'follower_friend_ratio', 'friends_to_followers', 'likes_to_tweets',
    'has_description', 'description_length', 'has_location', 'has_default_pic',
    'is_default_profile', 'is_verified', 'has_geo', 'profile_completeness',
    'is_high_tweeter', 'abnormal_tweet_rate', 'sudden_fame', 'engagement_paradox'
]

df_clean = df[FEATURES + ['label']].copy()
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
df_clean.fillna(0, inplace=True)
df_clean['label'] = df_clean['label'].astype(int)
df_clean.to_csv('data/processed_dataset.csv', index=False)

print('✅ Feature engineering complete!')
print(f'   Records  : {len(df_clean):,}')
print(f'   Features : {len(FEATURES)}')
print(f'   Bots     : {df_clean["label"].sum():,}  ({df_clean["label"].mean()*100:.1f}%)')
print(f'   Humans   : {(df_clean["label"]==0).sum():,}  ({(1-df_clean["label"].mean())*100:.1f}%)')
df_clean.describe().round(2)

## 🤖 Step 4 — Train Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.utils.class_weight import compute_class_weight
import joblib, json

X = df_clean[FEATURES]
y = df_clean['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Handle class imbalance
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
cw_dict = {0: cw[0], 1: cw[1]}
print(f'Class weights: {cw_dict}')

print('\n🌲 Training Random Forest (300 trees)...')
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight=cw_dict,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('✅ Training complete!')

## 📊 Step 5 — Evaluate Model

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy'  : round(accuracy_score(y_test, y_pred), 4),
    'precision' : round(precision_score(y_test, y_pred), 4),
    'recall'    : round(recall_score(y_test, y_pred), 4),
    'f1_score'  : round(f1_score(y_test, y_pred), 4),
    'roc_auc'   : round(roc_auc_score(y_test, y_prob), 4),
}

# 5-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
metrics['cv_auc_mean'] = round(cv_auc.mean(), 4)
metrics['cv_auc_std']  = round(cv_auc.std(), 4)

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('  MODEL EVALUATION RESULTS')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
for k, v in metrics.items():
    print(f'  {k:<16}: {v}')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

with open('outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('\n✅ Metrics saved to outputs/metrics.json')

## 📈 Step 6 — Generate All Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

DARK_BG = '#0D0F14'; CARD_BG = '#1A1D26'; BORDER = '#2A2D3A'
GREEN = '#4ECCA3'; RED = '#FF4757'; YELLOW = '#FFE66D'; GREY = '#8B8FA8'

def dark_fig(w=7, h=5):
    fig, ax = plt.subplots(figsize=(w, h), facecolor=DARK_BG)
    ax.set_facecolor(CARD_BG)
    for sp in ax.spines.values(): sp.set_color(BORDER)
    ax.tick_params(colors=GREY)
    ax.xaxis.label.set_color(GREY)
    ax.yaxis.label.set_color(GREY)
    ax.title.set_color('white')
    return fig, ax

# ── 1. Confusion Matrix ───────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = dark_fig(6, 5)
ax.imshow(cm, cmap='Blues', alpha=0.85)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Human','Bot'], fontsize=13, color='white')
ax.set_yticklabels(['Human','Bot'], fontsize=13, color='white')
ax.set_xlabel('Predicted', fontsize=13); ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix', fontsize=15, color='white', fontweight='bold', pad=14)
for i in range(2):
    for j in range(2):
        c = 'white' if cm[i,j] > cm.max()/2 else GREY
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center', fontsize=18, color=c, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved')

In [ ]:
# ── 2. ROC Curve ──────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = dark_fig(7, 5)
ax.plot(fpr, tpr, color=RED, lw=2.5, label=f'ROC AUC = {metrics["roc_auc"]:.4f}')
ax.fill_between(fpr, tpr, alpha=0.08, color=RED)
ax.plot([0,1],[0,1],'--',lw=1,color=BORDER)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Bot Detection', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, framealpha=0.2, facecolor=CARD_BG, labelcolor='white')
ax.grid(alpha=0.15, color=BORDER)
plt.tight_layout()
plt.savefig('outputs/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC curve saved')

In [ ]:
# ── 3. Feature Importance ─────────────────────────────────
feat_imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
top20 = feat_imp.tail(20)

fig, ax = dark_fig(11, 8)
colors = [RED if v > top20.median() else GREEN for v in top20.values]
bars = ax.barh(top20.index, top20.values, color=colors, height=0.7, edgecolor=BORDER, linewidth=0.5)
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Top 20 Behavioural Features\nDriving Bot vs Human Classification', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.15, color=BORDER)
for bar, val in zip(bars, top20.values):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, color=GREY)
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.savefig('outputs/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance saved')

In [ ]:
# ── 4. Feature Distributions: Bot vs Human ────────────────
top_feats = feat_imp.tail(6).index.tolist()[::-1]
fig, axes = plt.subplots(2, 3, figsize=(14, 8), facecolor=DARK_BG)
fig.suptitle('Behavioural Feature Distributions: Bot vs Human',
             fontsize=14, color='white', fontweight='bold', y=1.01)

df_full = pd.concat([X, y], axis=1)
for ax, feat in zip(axes.flat, top_feats):
    ax.set_facecolor(CARD_BG)
    for sp in ax.spines.values(): sp.set_color(BORDER)
    ax.hist(df_full[df_full['label']==0][feat], bins=40, alpha=0.7, color=GREEN, density=True, label='Human')
    ax.hist(df_full[df_full['label']==1][feat], bins=40, alpha=0.7, color=RED,   density=True, label='Bot')
    ax.set_title(feat.replace('_',' ').title(), fontsize=10, color='white')
    ax.tick_params(colors=GREY, labelsize=8)
    ax.legend(fontsize=8, framealpha=0.2, facecolor=CARD_BG, labelcolor='white')
    ax.grid(alpha=0.1, color=BORDER)

plt.tight_layout()
plt.savefig('outputs/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Distribution plots saved')

In [ ]:
# ── 5. Class Distribution Pie ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor=DARK_BG)

# Pie chart
ax = axes[0]
ax.set_facecolor(CARD_BG)
counts = y.value_counts()
ax.pie(counts, labels=['Human', 'Bot'], colors=[GREEN, RED],
       autopct='%1.1f%%', startangle=90,
       textprops={'color': 'white', 'fontsize': 13},
       wedgeprops={'edgecolor': DARK_BG, 'linewidth': 2})
ax.set_title('Dataset Composition', fontsize=13, color='white', fontweight='bold')

# Bar — top features summary
ax2 = axes[1]
ax2.set_facecolor(CARD_BG)
for sp in ax2.spines.values(): sp.set_color(BORDER)
top5 = feat_imp.tail(5)
ax2.barh(top5.index, top5.values, color=RED, height=0.6, edgecolor=BORDER)
ax2.set_title('Top 5 Bot Signals', fontsize=13, color='white', fontweight='bold')
ax2.tick_params(colors=GREY)
ax2.xaxis.label.set_color(GREY)
ax2.grid(axis='x', alpha=0.15, color=BORDER)
ax2.set_xlabel('Importance', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/summary_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Summary chart saved')

## 💾 Step 7 — Save Model & Feature List

In [ ]:
joblib.dump(model, 'models/rf_model.pkl')
pd.Series(FEATURES).to_csv('models/features.csv', index=False, header=False)

print('✅ Model saved   → models/rf_model.pkl')
print('✅ Features saved → models/features.csv')

# Final summary
print('\n' + '='*45)
print('  🎉 TRAINING PIPELINE COMPLETE!')
print('='*45)
print(f'  Accuracy   : {metrics["accuracy"]}')
print(f'  Precision  : {metrics["precision"]}')
print(f'  Recall     : {metrics["recall"]}')
print(f'  F1 Score   : {metrics["f1_score"]}')
print(f'  ROC AUC    : {metrics["roc_auc"]}')
print(f'  CV AUC     : {metrics["cv_auc_mean"]} ± {metrics["cv_auc_std"]}')
print('='*45)

print('\n  Files generated:')
for root, dirs, files in os.walk('.'):
    for file in files:
        if not file.startswith('.'):
            path = os.path.join(root, file)
            size = os.path.getsize(path)
            print(f'    {path}  ({size/1024:.1f} KB)')

## 📥 Step 8 — Download Everything as ZIP

In [ ]:
import shutil
from google.colab import files

# Zip the outputs and models folders
shutil.make_archive('fakescope_outputs', 'zip', '.', 'outputs')
shutil.make_archive('fakescope_models',  'zip', '.', 'models')
shutil.make_archive('fakescope_data',    'zip', '.', 'data')

print('Downloading outputs...')
files.download('fakescope_outputs.zip')

print('Downloading models...')
files.download('fakescope_models.zip')

print('Downloading processed data...')
files.download('fakescope_data.zip')

print('\n✅ All done! Unzip these into your project folder.')

---
## 🧪 Bonus — Quick Prediction Test
Test the model with a sample account manually.

In [ ]:
# Test with a clear BOT profile
bot_account = {
    'tweets_per_day': 120, 'account_age_days': 15, 'total_tweets': 1800,
    'total_likes_given': 0, 'tweet_density': 120, 'likes_per_day': 0,
    'followers_count': 5000, 'friends_count': 8000, 'follower_friend_ratio': 0.625,
    'friends_to_followers': 1.6, 'likes_to_tweets': 0,
    'has_description': 0, 'description_length': 0, 'has_location': 0,
    'has_default_pic': 1, 'is_default_profile': 1, 'is_verified': 0,
    'has_geo': 0, 'profile_completeness': 0,
    'is_high_tweeter': 1, 'abnormal_tweet_rate': 1, 'sudden_fame': 1, 'engagement_paradox': 1
}

# Test with a clear HUMAN profile
human_account = {
    'tweets_per_day': 3, 'account_age_days': 1200, 'total_tweets': 3600,
    'total_likes_given': 8000, 'tweet_density': 3, 'likes_per_day': 6.67,
    'followers_count': 850, 'friends_count': 620, 'follower_friend_ratio': 1.37,
    'friends_to_followers': 0.73, 'likes_to_tweets': 2.22,
    'has_description': 1, 'description_length': 95, 'has_location': 1,
    'has_default_pic': 0, 'is_default_profile': 0, 'is_verified': 0,
    'has_geo': 1, 'profile_completeness': 5,
    'is_high_tweeter': 0, 'abnormal_tweet_rate': 0, 'sudden_fame': 0, 'engagement_paradox': 0
}

for label, account in [('🤖 BOT ACCOUNT', bot_account), ('👤 HUMAN ACCOUNT', human_account)]:
    X_sample = pd.DataFrame([account])[FEATURES]
    prob_bot  = model.predict_proba(X_sample)[0][1]
    bot_score = round(prob_bot * 100, 1)
    auth_score = round(100 - bot_score, 1)
    verdict = 'BOT DETECTED' if bot_score>=70 else ('SUSPICIOUS' if bot_score>=40 else 'ORGANIC')
    print(f'\n{label}')
    print(f'  Bot Probability  : {bot_score}%')
    print(f'  Authenticity     : {auth_score}%')
    print(f'  Verdict          : {verdict}')